# M(EBA) Colab 원클릭 실행 노트북 (`run_all.ipynb`)

담당 B (PR B-5). 로직은 모두 `src/`·`scripts/run_eval.py`에 있고, 이 노트북은 **Colab T4 GPU** 환경에서 설치·검증·실행·결과 회수만 수행합니다.

- **시작 전:** Colab 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 저장하세요.
- **한 세션에 한 묶음/라운드만 실행**합니다 (`notebooks/README.md`, ADR-0005).
- **세션이 끊겼을 때:** 로그는 문항마다 `.jsonl`에 즉시 저장되므로, 런타임이 유지됐다면 4번·6번 셀을, 런타임이 초기화됐다면 1~6번 셀을 같은 `BUNDLE`·`ROUND`로 다시 실행하면 이어서 돕니다.
- **깨끗한 커밋 추적:** 모델/설정은 `configs/default.yaml`에서 읽고, `-B` 옵션으로 `__pycache__` 생성을 막아 `RUN_META.json`의 `git_dirty=false`를 유지합니다. 커밋 전에는 셀 출력을 지우세요.

## 0. GPU 확인 (`Tesla T4` 확인)

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 1. 실행 설정 (필요한 값만 바꿔 실행)

- **개발 세트 리허설 (`p0, p0_raw, p1, m`)**: `BUNDLE = "dev"`, `ROUND = "r8"` (기존 `r1`~`r7`과 겹치지 않는 새 라운드 번호)
- **본 비교 1라운드**: `BUNDLE = "main"`, `ROUND = "r1"` (`r1`~`r3`는 각각 새 Colab 세션 권장)
- **절제 1라운드**: `BUNDLE = "ablation"`, `ROUND = "r1"`, `PIPELINES = "m,m_minus_router,m_minus_requery,m_minus_judge,m_minus_compress,m_minus_cap,m_minus_loop_breaker"`

In [ ]:
import re

BRANCH = "main"              # 실험/리허설 브랜치명 (예: "main" 또는 "HyunSeong")
BUNDLE = "dev"               # "dev" | "main" | "ablation"
ROUND = "r8"                 # "r1", "r2", ... (소문자 r + 숫자)
PIPELINES = "p0,p0_raw,p1,m" # 비우면 main/dev 기본값(p0,p0_raw,p1,m). ablation은 필수 지정
ITEM_SET = ""                # 비우면 묶음 기본값 (dev->dev, main/ablation->eval)
ALLOW_DIRTY = False          # 원격에 push된 깨끗한 상태에서 돌릴 때 False 유지

assert BUNDLE in {"dev", "main", "ablation"}, f"지원하지 않는 BUNDLE: {BUNDLE!r}"
assert re.fullmatch(r"r\d+", ROUND), f"ROUND는 r1, r2 형식이어야 합니다: {ROUND!r}"
if BUNDLE == "ablation":
    assert PIPELINES.strip(), "ablation 묶음은 PIPELINES를 직접 지정해야 합니다."
print(f"설정 확인: BRANCH={BRANCH}, BUNDLE={BUNDLE}, ROUND={ROUND}, PIPELINES={PIPELINES or 'default'}, ITEM_SET={ITEM_SET or 'auto'}")

## 2. 코드 받기 (`{BRANCH}`)

In [ ]:
import os, subprocess

os.chdir("/content")
REPO_DIR = "/content/repo"
REPO_URL = "https://github.com/minyi-k03/Intelligence-Software-Project.git"
# 비공개 저장소면 Colab 비밀(userdata)에 GH_TOKEN을 넣고 아래 두 줄 주석을 해제하세요.
# from google.colab import userdata
# REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)

head_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
print("HEAD:", head_sha, "| dirty:", repr(dirty))
if not ALLOW_DIRTY:
    assert dirty == "", f"작업 트리가 깨끗하지 않습니다: {dirty!r}"

## 3. 의존성 설치

In [ ]:
!pip install -q -r requirements.txt && pip list 2>/dev/null | grep -iE "^(langchain|langgraph|langchain-ollama) "

## 4. Ollama 설치·서버 실행 및 모델 GPU 로드 확인

- `zstd`, `pciutils`, `lshw`를 먼저 설치해 Ollama가 T4 GPU를 인식하게 합니다.
- 모델 태그는 `configs/default.yaml`에서 읽고, `ollama ps`에서 `GPU` 적재를 확인합니다.

In [ ]:
import subprocess, time, urllib.request, yaml

subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd pciutils lshw > /dev/null"], check=True)
subprocess.run(["bash", "-lc", "which ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh"], check=True)

subprocess.run(["bash", "-lc", "pkill -f 'ollama serve' || true"], check=False)
time.sleep(2)

ollama_log = open("/content/ollama.log", "w", encoding="utf-8")
subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama 서버 준비됨")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama 서버가 60초 안에 응답하지 않았습니다. /content/ollama.log를 확인하세요.")

cfg = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))
tag = cfg["model"]["tag"]
print("model.tag =", tag)
pull = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
assert pull.returncode == 0, f"ollama pull 실패: {pull.stderr[-400:]}"

probe = subprocess.run(["ollama", "run", tag, "Say OK."], capture_output=True, text=True, timeout=300)
assert probe.returncode == 0, f"ollama run 실패: {probe.stderr[-400:]}"
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
assert "GPU" in ps and "CPU" not in ps.replace("PROCESSOR", ""), f"모델이 GPU에 올라가지 않았습니다:\n{ps}"
print("OK: 100% GPU 적재 확인")

## 5. 실행 전 검증 (동결 해시 + 설정 스냅샷)

In [ ]:
import sys
from pathlib import Path

sys.dont_write_bytecode = True
from src.config.settings import get_settings
from src.eval.logger import verify_frozen_hashes

hashes = verify_frozen_hashes(Path("data/frozen"), Path("data/README.md"))
print("동결 해시 일치:", {k: v[:12] for k, v in hashes.items()})
s = get_settings()
print(
    f"설정: model={s.model.tag}, temp={s.model.temperature}, num_ctx={s.model.num_ctx}, "
    f"keep_alive={s.model.keep_alive}, k={s.retrieval.k}, m={s.compress.m}, judge.format={s.judge.format}"
)

## 6. 본 실행 (`scripts.run_eval`)

중단되면 같은 설정으로 이 셀만 다시 실행하면 미완료 문항부터 이어서 돕니다 (`RunMeta`가 설정·해시·실행 순서 일치를 자동 검사).

In [ ]:
import subprocess, time

cmd = ["python3", "-B", "-m", "scripts.run_eval", "--bundle", BUNDLE, "--round", ROUND]
if PIPELINES.strip():
    cmd.extend(["--pipelines", PIPELINES.strip()])
if ITEM_SET.strip():
    cmd.extend(["--set", ITEM_SET.strip()])

print("실행 명령:", " ".join(cmd))
t0 = time.monotonic()
subprocess.run(cmd, check=True)
elapsed = time.monotonic() - t0
print(f"\n총 실행 시간: {elapsed:.1f}s ({elapsed / 60:.1f}분)")

## 7. 라운드 요약 확인 (`RUN_META.json` + 파이프라인별 지표)

In [ ]:
import json, statistics
from pathlib import Path

out_dir = Path("results/runs") / BUNDLE / ROUND
meta = json.loads((out_dir / "RUN_META.json").read_text(encoding="utf-8"))
print("=== RUN_META ===")
print("git_commit:", meta.get("git_commit"), "| git_dirty:", meta.get("git_dirty"))
print("pipeline_order:", meta.get("pipeline_order"))
print("max_vram_mib_by_pipeline:", meta.get("max_vram_mib_by_pipeline"), "| max:", meta.get("max_vram_mib"))
print("sessions:", len(meta.get("sessions", [])))

print("\n=== 파이프라인별 요약 (비용·시간은 워밍업 1문항 제외) ===")
for name in meta.get("pipeline_order", []):
    rows = [json.loads(l) for l in (out_dir / f"{name}.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    cost = [r for r in rows if not r["is_warmup"]]
    by_type = {
        t: f"{sum(1 for r in rows if r['type'] == t and r['correct'])}/{sum(1 for r in rows if r['type'] == t)}"
        for t in ("hotpotqa", "squad2", "gsm8k")
    }
    n_err = sum(1 for r in rows if r.get("error"))
    n_to = sum(1 for r in rows if r.get("timeout"))
    n_ex = sum(1 for r in rows if r.get("budget_exhausted"))
    lats = [r["latency_s"] for r in cost]
    calls = [r["n_calls"] for r in cost]
    out_toks = [sum(c["output_tokens"] for c in r["calls"]) for r in cost]
    print(
        f"[{name:20s}] n={len(rows)} acc={sum(r['correct'] for r in rows)}/{len(rows)} "
        f"({sum(r['correct'] for r in rows)/len(rows):.1%}) {by_type} | "
        f"calls={sum(calls)/len(calls):.2f} out_tok={sum(out_toks)/len(out_toks):.1f} "
        f"lat_mean={sum(lats)/len(lats):.2f}s lat_med={statistics.median(lats):.2f}s | "
        f"err={n_err} to={n_to} exh={n_ex}"
    )

## 8. 결과 내려받기 (`{BUNDLE}_{ROUND}.zip`)

브라우저 Colab에서는 자동 다운로드가 뜨며, VS Code Colab 확장처럼 다운로드를 지원하지 않는 환경에서는 아래 zip 경로 또는 base64 출력으로 로컬에 저장해 `results/runs/{BUNDLE}/{ROUND}/`에 넣습니다.

In [ ]:
import base64, pathlib, shutil

zip_base = f"/content/{BUNDLE}_{ROUND}"
zip_path = shutil.make_archive(zip_base, "zip", f"results/runs/{BUNDLE}/{ROUND}")
print("생성됨:", zip_path, f"({pathlib.Path(zip_path).stat().st_size} bytes)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("자동 다운로드 불가(VS Code 등):", exc)
    print("필요 시 아래 주석을 풀어 base64로 복사하세요:")
    # print(base64.b64encode(pathlib.Path(zip_path).read_bytes()).decode())